# Live Demo: Calling an LLM API from Python

This notebook demonstrates the following elements:

- making a first LLM API call
- understanding the API response
- extracting the generated answer
- inspecting token usage and cost-related metadata
- seeing how parameters such as `temperature` change behavior

The goal is not to teach every OpenAI API feature. The goal is to show the basic developer workflow: **build request → send request → inspect response → use result**.

## 1. Setup

We use the OpenAI Python SDK. The API key should be stored as an environment variable named `OPENAI_API_KEY`.

That means we do **not** hardcode secrets in the notebook.

On macOS or Linux, you can set it in a terminal with:

```bash
export OPENAI_API_KEY="your_api_key_here"
```

Then launch Jupyter from that same terminal. Alternatively store your kesy in an keys.env file, and load it as we do in the next cell.

In [2]:
# If needed, install the SDK once:
#!pip install openai

import truststore
truststore.inject_into_ssl()

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("keys.env")

client = OpenAI()

print("OpenAI client initialized.")
print("API key found:", bool(os.getenv("OPENAI_API_KEY")))

OpenAI client initialized.
API key found: True


## 2. First API Call

This is the simplest possible useful call:

- choose a model
- send a user message
- receive a response

For the class demo, we use a networking-related example: **Wi-Fi roaming**.

In [10]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Explain Wi-Fi roaming in 2 sentences for a junior network engineer."
        }
    ],
)

print(response.choices[0].message.content)

Wi-Fi roaming allows a device to seamlessly switch between different access points (APs) within the same wireless network without losing connectivity, which is essential for maintaining an uninterrupted connection as users move around. This process typically involves the device associating with a new AP that has a stronger signal while staying connected to the same network, facilitated by standards like 802.11r for faster transitions.


In [13]:
#Ollama local version
import requests
from pprint import pprint

r = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:latest",
        "prompt": "Explain Wi-Fi roaming in 2 sentences for a junior network engineer.",
        "stream": False
    }
)

print("Status:", r.status_code)
pprint(r.json())

Status: 200
{'context': [128006,
             882,
             128007,
             271,
             849,
             21435,
             17664,
             27395,
             69353,
             304,
             220,
             17,
             23719,
             369,
             264,
             27144,
             4009,
             24490,
             13,
             128009,
             128006,
             78191,
             128007,
             271,
             8586,
             596,
             459,
             16540,
             315,
             17664,
             27395,
             69353,
             304,
             220,
             17,
             23719,
             1473,
             59128,
             27395,
             69353,
             6276,
             6505,
             7766,
             311,
             61440,
             3480,
             1990,
             2204,
             21401,
             2680,
             3585,
           

### What just happened?

Your Python code created a structured request and sent it to the model provider.

Conceptually:

```text
Your notebook → API request → hosted model → API response → your notebook
```

The important point is that the model response is not just a plain string. It comes back as a structured object.

## 3. Inspect the Full Response Object

In a real application, you often need more than the answer text.

The full response can include:

- generated message content
- model name
- finish reason
- token usage
- response metadata

In [14]:
# Convert the SDK object to a plain Python dictionary for easier inspection.
response_dict = response.model_dump()

pprint(response_dict)

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'annotations': [],
                          'audio': None,
                          'content': 'Wi-Fi roaming allows a device to '
                                     'seamlessly switch between different '
                                     'access points (APs) within the same '
                                     'wireless network without losing '
                                     'connectivity, which is essential for '
                                     'maintaining an uninterrupted connection '
                                     'as users move around. This process '
                                     'typically involves the device '
                                     'associating with a new AP that has a '
                                     'stronger signal while staying connected '
                                     'to the same network, fa

## 4. Extract the Fields You Actually Use

Most applications extract a few key fields:

- the generated answer
- token usage
- finish reason

The answer goes to the user or downstream system. The usage metadata helps with cost monitoring and optimization.

In [17]:
answer = response.choices[0].message.content
finish_reason = response.choices[0].finish_reason
usage = response.usage

print("Answer:", answer)
print("Finish reason:", finish_reason)
print("Usage:")
print("Prompt tokens:", usage.prompt_tokens)
print("Completion tokens:", usage.completion_tokens)
print("Total tokens:", usage.total_tokens)

Answer: Wi-Fi roaming allows a device to seamlessly switch between different access points (APs) within the same wireless network without losing connectivity, which is essential for maintaining an uninterrupted connection as users move around. This process typically involves the device associating with a new AP that has a stronger signal while staying connected to the same network, facilitated by standards like 802.11r for faster transitions.
Finish reason: stop
Usage:
Prompt tokens: 21
Completion tokens: 77
Total tokens: 98


### Point to note:

The API response gives you both:

1. **The useful output** — the generated answer.
2. **Operational metadata** — token usage, finish reason, and other fields.

That metadata becomes important later for monitoring, debugging, cost control, and reliability.

## 5. Wrap the Call in a Function

In a real application, you rarely scatter raw API calls throughout your code.

Instead, you wrap the call in a function so that the rest of your application has a clean interface.

In [18]:
def ask_llm(question: str, model: str = "gpt-4o-mini", temperature: float = 0.2) -> dict:
    """
    Send a question to the LLM and return a small, application-friendly result.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful technical teaching assistant. Be accurate and concise."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=temperature,
    )

    return {
        "answer": response.choices[0].message.content,
        "finish_reason": response.choices[0].finish_reason,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
        "model": response.model,
    }

result = ask_llm("Explain why Wi-Fi clients roam between access points.")
pprint(result)

{'answer': 'Wi-Fi clients roam between access points (APs) primarily to '
           'maintain a strong and stable connection as they move through an '
           'area covered by multiple APs. Here are the key reasons for this '
           'behavior:\n'
           '\n'
           '1. **Signal Strength**: As a client moves away from an AP, the '
           'signal strength decreases. Roaming allows the client to connect to '
           'a closer AP with a stronger signal, ensuring better performance '
           'and reliability.\n'
           '\n'
           '2. **Load Balancing**: In environments with multiple APs, clients '
           'may roam to distribute the network load more evenly. This helps '
           'prevent any single AP from becoming overloaded, which can degrade '
           'performance for all users.\n'
           '\n'
           '3. **Network Optimization**: Some networks are designed to '
           'optimize performance by encouraging clients to connect to the mo

### Why this wrapper matters

This small function is the beginning of application design.

It gives you one place to control:

- the model
- the system prompt
- parameters such as temperature
- logging
- error handling
- usage tracking

This is how you move from a notebook experiment toward a reusable backend service.

## 6. Parameter Demo: Temperature

The `temperature` parameter changes how varied or creative the output can be.

A lower value usually makes the model more focused and consistent. A higher value allows more variation.

For technical applications, low or moderate values are often preferred.

In [22]:
question = "Explain Wi-Fi roaming in one short paragraph."

low_temp = ask_llm(question, temperature=0.1)
higher_temp = ask_llm(question, temperature=0.9)
very_high_temp = ask_llm(question, temperature=1.9)

print("=== Low temperature: 0.1 ===")
print(low_temp["answer"])
print("Tokens:", low_temp["total_tokens"])

print("=== Higher temperature: 0.9 ===")
print(higher_temp["answer"])
print("Tokens:", higher_temp["total_tokens"])

print("=== Very high temperature: 1.9 ===")
print(very_high_temp["answer"])
print("Tokens:", very_high_temp["total_tokens"])

=== Low temperature: 0.1 ===
Wi-Fi roaming refers to the ability of a device to maintain a continuous internet connection while moving between different access points within the same network without experiencing interruptions. This is achieved through seamless handoff mechanisms that allow the device to switch from one access point to another, typically using the same SSID (network name), ensuring a stable connection as users move throughout a coverage area, such as in large buildings or campuses.
Tokens: 114
=== Higher temperature: 0.9 ===
Wi-Fi roaming refers to the ability of a device to maintain a seamless internet connection as it moves between different access points within a wireless network. This process typically involves the device automatically switching to a stronger signal from a nearby access point without requiring user intervention, thus ensuring continuous connectivity and a smooth user experience. Roaming is often facilitated by technologies such as 802.11r, which ena

### Point to note:

Changing parameters changes behavior. AI applications are not controlled only through code. They are shaped through:

- prompts
- model choice
- parameters
- retrieved context
- system design

## 7. Optional: Ask for Structured JSON Output

Many real applications do not want free-form text. They want structured output that another program can parse.

This example asks the model to return JSON. In production, you would usually add stronger validation with Pydantic or a provider-specific structured output feature.

In [24]:
structured_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Return only valid JSON. Do not include Markdown."
        },
        {
            "role": "user",
            "content": "Analyze this issue: A laptop drops a video call when walking between rooms. Return fields: issue_summary, likely_cause, next_step."
        }
    ],
    temperature=0.2,
)

json_text = structured_response.choices[0].message.content
print(json_text)

# Try to parse it as JSON.
parsed = json.loads(json_text)
print("Parsed object:")
pprint(parsed)

{
  "issue_summary": "The laptop disconnects from a video call when the user moves between rooms.",
  "likely_cause": "Weak Wi-Fi signal or interference due to distance from the router or physical barriers (walls, furniture) affecting connectivity.",
  "next_step": "Check the Wi-Fi signal strength in different rooms and consider moving closer to the router or using a Wi-Fi extender to improve connectivity."
}
Parsed object:
{'issue_summary': 'The laptop disconnects from a video call when the user '
                  'moves between rooms.',
 'likely_cause': 'Weak Wi-Fi signal or interference due to distance from the '
                 'router or physical barriers (walls, furniture) affecting '
                 'connectivity.',
 'next_step': 'Check the Wi-Fi signal strength in different rooms and consider '
              'moving closer to the router or using a Wi-Fi extender to '
              'improve connectivity.'}


### Point to note:

Structured output is important because most AI applications are not just chatbots.

They often need to feed results into:

- dashboards
- databases
- APIs
- automation workflows
- evaluation pipelines

## 8. Minimal Error Handling Pattern

Production code needs to handle API failures, timeouts, rate limits, and invalid outputs.

This is a minimal demonstration. Full codes typically expand this with retries, logging, and observability.

In [25]:
def safe_ask_llm(question: str) -> dict:
    try:
        return ask_llm(question)
    except Exception as exc:
        return {
            "answer": None,
            "error": str(exc),
        }

safe_result = safe_ask_llm("Explain what an LLM API is in one sentence.")
pprint(safe_result)

{'answer': 'An LLM API (Large Language Model Application Programming '
           'Interface) is a set of protocols that allows developers to '
           'integrate and interact with large language models for tasks such '
           'as text generation, summarization, and natural language '
           'understanding.',
 'completion_tokens': 43,
 'finish_reason': 'stop',
 'model': 'gpt-4o-mini-2024-07-18',
 'prompt_tokens': 35,
 'total_tokens': 78}


## 9. Demo Wrap-Up

In this short demo, we saw the core developer workflow:

1. Initialize the client.
2. Send a structured request.
3. Receive a structured response.
4. Extract the answer.
5. Inspect usage metadata.
6. Wrap the call in reusable code.
7. Control behavior through parameters.
8. Optionally request structured output.

The key idea is simple:

> An LLM API is a software interface to a model. The model may be powerful, but your application still needs clean engineering around it.

## Good vs Bad: Raw API Calls vs Wrapper

The notebook started with raw API calls because they are easy to understand.

But in a real application, we usually do **not** want model calls scattered throughout the code.

A wrapper gives us one controlled place to manage:

- model choice
- parameters
- logging
- errors
- retries
- provider switching

In [3]:
# BAD PATTERN: raw model calls scattered throughout the application

response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain Wi-Fi roaming in 2 sentences."}
    ],
    temperature=0.7,
    max_tokens=120,
)

print(response1.choices[0].message.content)


response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Summarize why retries matter for LLM APIs."}
    ],
    temperature=0.7,
    max_tokens=120,
)

print(response2.choices[0].message.content)

Wi-Fi roaming allows a device to maintain its internet connection as it moves between different access points within the same network without needing to manually reconnect. This seamless transition enhances user experience by providing continuous connectivity, especially in larger areas like offices or public spaces where multiple access points are deployed.
Retries are important for LLM (Large Language Model) APIs for several reasons:

1. **Reliability**: Network issues, server overloads, or temporary outages can lead to failed requests. Implementing retries ensures that transient errors do not disrupt the user experience.

2. **Error Handling**: LLM APIs may return errors due to various reasons, including rate limits or service unavailability. Retrying can help recover from these errors without requiring user intervention.

3. **Consistency**: In cases where the output may vary slightly with each request, retries can help ensure that users receive a satisfactory response


### What is wrong with this?

This works, but it does not scale well.

If we want to change the model, temperature, logging, retries, or error handling, we now have to update many places in the code.

That creates duplicated logic and fragile applications.

In [4]:
def ask_llm(
    prompt: str,
    model: str = "gpt-4o-mini",
    temperature: float = 0.7,
    max_tokens: int = 120,
) -> str:
    """
    Simple wrapper around the LLM API.

    This centralizes model access so the rest of the application
    does not call the provider directly.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content

In [5]:
answer = ask_llm("Explain Wi-Fi roaming in 2 sentences.")
print(answer)

Wi-Fi roaming allows a device to maintain a continuous internet connection while moving between different access points within the same network, without needing to reconnect manually. This seamless transition enhances user experience by providing uninterrupted connectivity, especially in larger environments like offices or campuses.


In [6]:
answer = ask_llm("Summarize why retries matter for LLM APIs.")
print(answer)

Retries are important for LLM (Large Language Model) APIs for several reasons:

1. **Network Reliability**: APIs can experience transient network issues that may result in failed requests. Implementing retries helps ensure that temporary problems do not lead to lost opportunities for processing data.

2. **Rate Limiting**: LLM APIs often have rate limits. If a request exceeds these limits, it may be rejected. Retrying with backoff strategies can help manage the load and adhere to API usage policies.

3. **Server Overload**: High demand can lead to server-side failures or timeouts.


In [7]:
# Even better wrapper, with metadata
def ask_llm_with_metadata(
    prompt: str,
    model: str = "gpt-4o-mini",
    temperature: float = 0.7,
    max_tokens: int = 120,
) -> dict:
    """
    Wrapper that returns both the answer and useful metadata.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return {
        "answer": response.choices[0].message.content,
        "model": model,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }

In [8]:
result = ask_llm_with_metadata(
    "Explain why token usage matters in LLM applications."
)

result

{'answer': 'Token usage is a crucial aspect of working with large language models (LLMs) due to several reasons:\n\n1. **Cost Management**: Many LLM providers charge based on the number of tokens processed. Understanding token usage helps users manage costs effectively, especially in applications requiring high volumes of text processing, such as customer support chatbots or content generation.\n\n2. **Performance and Speed**: The number of tokens can impact the performance of LLMs. More tokens generally mean longer processing times. Optimizing token usage can lead to faster responses, which is particularly important in real-time applications like live chat',
 'model': 'gpt-4o-mini',
 'prompt_tokens': 17,
 'completion_tokens': 120,
 'total_tokens': 137}

In [9]:
# Even  better, a version that allows you to switch API provider
LLM_PROVIDER = "openai"  # later this could be "ollama", "anthropic", etc.

def ask_model(prompt: str) -> str:
    """
    Conceptual provider wrapper.

    The rest of the application calls ask_model().
    Only this function needs to know which provider is being used.
    """

    if LLM_PROVIDER == "openai":
        return ask_llm(prompt)

    raise ValueError(f"Unsupported provider: {LLM_PROVIDER}")

### Key takeaway

A wrapper is not just about reducing code.

It creates a stable boundary between the application and the model provider.

That boundary is where we can add reliability, logging, cost tracking, retries, fallbacks, and provider switching.